# PA1 Task 2 — Autoformer leaderboard forecaster (168 steps)
##Name: Sadaf Aamer
##Roll Number: 25280007

This notebook keeps the **original outputs** of the Task 2 experiments, run on a Colab T4 GPU.
Cells were run interactively and in a different order during development; they are grouped here
in report order with their outputs unchanged. Code that later changed between runs is preserved
in its final form. For a clean, top-to-bottom reproduction of every result, use
`autoformer_task2_clean.ipynb`.

The tuning and routing grids (report Tables 3–4) were run in earlier iterations of this notebook, whose outputs were overwritten; they are reproduced by sections C–D of the clean notebook.




In [ ]:
import argparse
import json
import math
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

N_TOTAL = 43824


## Autoformer building blocks

In [ ]:
class MovingAvg(nn.Module):
    """Centered moving average with endpoint-replication padding (keeps length)."""

    def __init__(self, kernel_size):
        super().__init__()
        assert kernel_size % 2 == 1, "moving_avg kernel must be odd"
        self.k = kernel_size
        self.avg = nn.AvgPool1d(kernel_size, stride=1, padding=0)

    def forward(self, x):  # x: [B, L, D]
        q = (self.k - 1) // 2
        front = x[:, :1, :].repeat(1, q, 1)
        end = x[:, -1:, :].repeat(1, q, 1)
        x = torch.cat([front, x, end], dim=1)
        return self.avg(x.permute(0, 2, 1)).permute(0, 2, 1)


class SeriesDecomp(nn.Module):
    """x = seasonal + trend, trend = moving average."""

    def __init__(self, kernel_size):
        super().__init__()
        self.ma = MovingAvg(kernel_size)

    def forward(self, x):
        trend = self.ma(x)
        return x - trend, trend


class SeasonalLayerNorm(nn.Module):
    """my_Layernorm: LayerNorm then remove the per-series mean over time."""

    def __init__(self, d):
        super().__init__()
        self.ln = nn.LayerNorm(d)

    def forward(self, x):
        x_hat = self.ln(x)
        return x_hat - x_hat.mean(dim=1, keepdim=True)


class AutoCorrelation(nn.Module):
    """Period-based dependency discovery (FFT) + time-delay aggregation.

    R(tau) = sum_t q_t k_{t-tau}, computed for all tau in O(L log L) via FFT.
    The top-k delays (k = int(factor * ln L)) are softmax-weighted and the value
    sequence is rolled by each delay and aggregated.
    """

    def __init__(self, factor=1.0, dropout=0.0):
        super().__init__()
        self.factor = factor
        self.dropout = nn.Dropout(dropout)

    def time_delay_agg(self, values, corr):
        # values, corr: [B, H, E, L]
        B, H, E, L = values.shape
        top_k = max(1, int(self.factor * math.log(L)))
        mean_corr = corr.mean(dim=1).mean(dim=1)                      # [B, L]
        weights, delay = torch.topk(mean_corr, top_k, dim=-1)        # [B, k]
        w = torch.softmax(weights, dim=-1)
        w = self.dropout(w)
        init_index = torch.arange(L, device=values.device).view(1, 1, 1, L).expand(B, H, E, L)
        tmp_values = values.repeat(1, 1, 1, 2)                        # circular via doubling
        agg = torch.zeros_like(values)
        for i in range(top_k):
            idx = init_index + delay[:, i].view(B, 1, 1, 1)
            pattern = torch.gather(tmp_values, dim=-1, index=idx)
            agg = agg + pattern * w[:, i].view(B, 1, 1, 1)
        return agg

    def forward(self, queries, keys, values):
        # [B, L, H, E] / [B, S, H, E]
        B, L, H, E = queries.shape
        S = values.shape[1]
        if L > S:
            zeros = torch.zeros_like(queries[:, : (L - S)])
            values = torch.cat([values, zeros], dim=1)
            keys = torch.cat([keys, zeros], dim=1)
        else:
            values = values[:, :L]
            keys = keys[:, :L]
        q_fft = torch.fft.rfft(queries.permute(0, 2, 3, 1).contiguous(), dim=-1)
        k_fft = torch.fft.rfft(keys.permute(0, 2, 3, 1).contiguous(), dim=-1)
        corr = torch.fft.irfft(q_fft * torch.conj(k_fft), n=L, dim=-1)   # [B, H, E, L]
        out = self.time_delay_agg(values.permute(0, 2, 3, 1).contiguous(), corr)
        return out.permute(0, 3, 1, 2).contiguous()                     # [B, L, H, E]


class AutoCorrelationLayer(nn.Module):
    def __init__(self, d_model, n_heads, factor, dropout):
        super().__init__()
        dk = d_model // n_heads
        self.h = n_heads
        self.corr = AutoCorrelation(factor, dropout)
        self.q = nn.Linear(d_model, dk * n_heads)
        self.k = nn.Linear(d_model, dk * n_heads)
        self.v = nn.Linear(d_model, dk * n_heads)
        self.o = nn.Linear(dk * n_heads, d_model)

    def forward(self, q, k, v):
        B, L, _ = q.shape
        S = k.shape[1]
        q = self.q(q).view(B, L, self.h, -1)
        k = self.k(k).view(B, S, self.h, -1)
        v = self.v(v).view(B, S, self.h, -1)
        return self.o(self.corr(q, k, v).reshape(B, L, -1))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, factor, ma, dropout):
        super().__init__()
        self.attn = AutoCorrelationLayer(d_model, n_heads, factor, dropout)
        self.conv1 = nn.Conv1d(d_model, d_ff, 1, bias=False)
        self.conv2 = nn.Conv1d(d_ff, d_model, 1, bias=False)
        self.decomp1 = SeriesDecomp(ma)
        self.decomp2 = SeriesDecomp(ma)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(x, x, x))
        x, _ = self.decomp1(x)                       # discard trend, keep seasonal
        y = self.drop(F.gelu(self.conv1(x.transpose(1, 2))))
        y = self.drop(self.conv2(y).transpose(1, 2))
        x, _ = self.decomp2(x + y)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, c_out, n_heads, d_ff, factor, ma, dropout):
        super().__init__()
        self.self_attn = AutoCorrelationLayer(d_model, n_heads, factor, dropout)
        self.cross_attn = AutoCorrelationLayer(d_model, n_heads, factor, dropout)
        self.conv1 = nn.Conv1d(d_model, d_ff, 1, bias=False)
        self.conv2 = nn.Conv1d(d_ff, d_model, 1, bias=False)
        self.decomp1 = SeriesDecomp(ma)
        self.decomp2 = SeriesDecomp(ma)
        self.decomp3 = SeriesDecomp(ma)
        self.drop = nn.Dropout(dropout)
        self.trend_proj = nn.Conv1d(d_model, c_out, 3, padding=1, padding_mode="circular", bias=False)

    def forward(self, x, cross):
        x = x + self.drop(self.self_attn(x, x, x))
        x, t1 = self.decomp1(x)
        x = x + self.drop(self.cross_attn(x, cross, cross))
        x, t2 = self.decomp2(x)
        y = self.drop(F.gelu(self.conv1(x.transpose(1, 2))))
        y = self.drop(self.conv2(y).transpose(1, 2))
        x, t3 = self.decomp3(x + y)
        trend = self.trend_proj((t1 + t2 + t3).permute(0, 2, 1)).transpose(1, 2)
        return x, trend


class DataEmbedding(nn.Module):
    """Value embedding (circular conv) + linear embedding of marks/covariates. No positional encoding."""

    def __init__(self, c_in, mark_dim, d_model, dropout):
        super().__init__()
        self.tok = nn.Conv1d(c_in, d_model, 3, padding=1, padding_mode="circular", bias=False)
        self.mark = nn.Linear(mark_dim, d_model, bias=False) if mark_dim > 0 else None
        self.drop = nn.Dropout(dropout)

    def forward(self, x, m):
        h = self.tok(x.permute(0, 2, 1)).transpose(1, 2)
        if self.mark is not None:
            h = h + self.mark(m)
        return self.drop(h)


class Autoformer(nn.Module):
    def __init__(self, seq_len, label_len, pred_len, enc_mark_dim, dec_mark_dim,
                 d_model=32, n_heads=4, e_layers=1, d_layers=1, d_ff=64,
                 moving_avg=25, factor=1.0, dropout=0.05, n_cov=0):
        super().__init__()
        self.seq_len, self.label_len, self.pred_len = seq_len, label_len, pred_len
        self.n_cov = n_cov                      # covariates fed as extra input channels
        self.revin = True
        self.decomp = SeriesDecomp(moving_avg)
        self.enc_emb = DataEmbedding(1 + n_cov, enc_mark_dim - n_cov, d_model, dropout)
        self.dec_emb = DataEmbedding(1 + n_cov, dec_mark_dim - n_cov, d_model, dropout)
        self.encoder = nn.ModuleList(
            [EncoderLayer(d_model, n_heads, d_ff, factor, moving_avg, dropout) for _ in range(e_layers)])
        self.enc_norm = SeasonalLayerNorm(d_model)
        self.decoder = nn.ModuleList(
            [DecoderLayer(d_model, 1, n_heads, d_ff, factor, moving_avg, dropout) for _ in range(d_layers)])
        self.dec_norm = SeasonalLayerNorm(d_model)
        self.proj = nn.Linear(d_model, 1)

    def forward(self, x_enc, m_enc, m_dec):
        # In channel mode the last n_cov columns of the marks are the covariates;
        # split them off and feed them as extra value channels.
        k = self.n_cov
        if k > 0:
            c_enc, m_enc = m_enc[..., -k:], m_enc[..., :-k]
            c_dec, m_dec = m_dec[..., -k:], m_dec[..., :-k]
        if self.revin:                          # normalise the target channel only
            mu = x_enc.mean(dim=1, keepdim=True).detach()
            sd = (x_enc.std(dim=1, keepdim=True) + 1e-5).detach()
            x_enc = (x_enc - mu) / sd

        B = x_enc.shape[0]
        mean = x_enc.mean(dim=1, keepdim=True).repeat(1, self.pred_len, 1)
        zeros = torch.zeros(B, self.pred_len, 1, device=x_enc.device)
        seasonal_init, trend_init = self.decomp(x_enc)
        trend = torch.cat([trend_init[:, -self.label_len:], mean], dim=1)
        seasonal = torch.cat([seasonal_init[:, -self.label_len:], zeros], dim=1)  # target unknown on horizon

        enc_in = torch.cat([x_enc, c_enc], -1) if k > 0 else x_enc
        dec_in = torch.cat([seasonal, c_dec], -1) if k > 0 else seasonal  # real future covariates

        enc = self.enc_emb(enc_in, m_enc)
        for layer in self.encoder:
            enc = layer(enc)
        enc = self.enc_norm(enc)

        dec = self.dec_emb(dec_in, m_dec)
        for layer in self.decoder:
            dec, rt = layer(dec, enc)
            trend = trend + rt
        seasonal_out = self.proj(self.dec_norm(dec))

        out = (trend + seasonal_out)[:, -self.pred_len:, :]
        return out * sd + mu if self.revin else out


def count_params(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


## Data

In [ ]:
def load_raw(args):
    tr = pd.read_csv(os.path.join(args.data_dir, args.train_file)).sort_values("time_idx").reset_index(drop=True)
    ex = pd.read_csv(os.path.join(args.data_dir, args.external_file)).sort_values("time_idx").reset_index(drop=True)
    y = tr["value"].to_numpy(dtype=float)
    assert not np.isnan(y).any(), "training target contains NaNs"
    assert len(ex) == N_TOTAL, f"external file has {len(ex)} rows, expected {N_TOTAL}"
    assert np.array_equal(ex["time_idx"].to_numpy()[: len(y)], tr["time_idx"].to_numpy()), \
        "time_idx of train and external files do not line up"
    cols = [c for c in ex.columns if c != "time_idx"]
    if args.features != "all":
        keep = [c.strip() for c in args.features.split(",") if c.strip()]
        cols = [c for c in cols if c in keep]
    X = ex[cols].astype(float)
    if X.isna().any().any():
        print("[warn] NaNs in external data -> forward/backward filled")
        X = X.ffill().bfill()
    return y, X.to_numpy(), cols, ex["time_idx"].to_numpy(dtype=float)


def detect_periods(y, top=2, min_p=4, max_p=400):
    """Recover periodicities from the training series only.

    log1p tames the heavy tail; subtracting a 337-step centered moving average removes
    slow (multi-week) wandering so it does not swamp the periodogram; peaks are ranked
    by prominence (power / local median power) and harmonics within 10% are merged.
    """
    x = np.log1p(np.clip(y, 0, None))
    w = 337
    slow = pd.Series(x).rolling(w, center=True, min_periods=1).mean().to_numpy()
    x = x - slow
    x = x - x.mean()
    spec = np.abs(np.fft.rfft(x)) ** 2
    freqs = np.fft.rfftfreq(len(x))
    spec, freqs = spec[1:], freqs[1:]
    periods = 1.0 / freqs
    local_med = pd.Series(spec).rolling(101, center=True, min_periods=1).median().to_numpy()
    prom = spec / (local_med + 1e-12)
    mask = (periods >= min_p) & (periods <= max_p)
    order = np.argsort(prom[mask])[::-1]
    cand_p, cand_r = periods[mask][order], prom[mask][order]
    chosen = []
    for p, r in zip(cand_p, cand_r):
        pr = int(round(p))
        if all(abs(pr - c) / c > 0.1 for c, _ in chosen):
            chosen.append((pr, float(r)))
        if len(chosen) == top:
            break
    return chosen


class Prep:
    """Scaled target + encoder/decoder mark matrices, built for one exog mode.

    All statistics are fitted on [0, fit_end) only (the training region before validation).
    """

    def __init__(self, y, X, cols, tidx, periods, args, mode, device):
        self.n = len(y)
        self.fit_end = self.n - args.val_len if args.val_len > 0 else self.n
        self.transform = args.target_transform
        zt = np.log1p(np.clip(y, 0, None)) if self.transform == "log1p" else y.copy()
        self.mu = float(zt[: self.fit_end].mean())
        self.sd = float(zt[: self.fit_end].std() + 1e-8)
        self.z = torch.tensor((zt - self.mu) / self.sd, dtype=torch.float32, device=device)

        # cyclical features of the recovered periods (derived from the integer index only)
        cyc = []
        for p in periods:
            cyc += [np.sin(2 * np.pi * tidx / p), np.cos(2 * np.pi * tidx / p)]
        cyc = np.stack(cyc, 1) if cyc else np.zeros((N_TOTAL, 0))

        # covariates: binary kept as is; skewed non-negative continuous -> log1p; then standardise
        ex = np.zeros((N_TOTAL, 0))
        if mode != "none" and X.shape[1] > 0:
            ex = X.copy()
            for j in range(ex.shape[1]):
                col = ex[:, j]
                if set(np.unique(col)).issubset({0.0, 1.0}):
                    continue
                if col.min() >= 0 and pd.Series(col[: self.fit_end]).skew() > 2:
                    col = np.log1p(col)
                m, s = col[: self.fit_end].mean(), col[: self.fit_end].std() + 1e-8
                ex[:, j] = (col - m) / s


        for c, k in getattr(args, "shifts", []):
          col = ex[:, cols.index(c)]
          sh = np.roll(col, k)
          if k > 0: sh[:k] = col[0]
          elif k < 0: sh[k:] = col[-1]
          ex = np.concatenate([ex, sh[:, None]], 1)

        for w in getattr(args, "roll_bin", []):
          for j in range(len(cols)):
            col = ex[:, j]
            if set(np.unique(col)).issubset({0.0, 1.0}):
              r = pd.Series(col).rolling(w, min_periods=1).mean().to_numpy()
              ex = np.concatenate([ex, r[:, None]], 1)
        self.mode, self.n_cov = mode, ex.shape[1]
        enc = np.concatenate([cyc, ex], 1) if mode in ("past", "future", "ch") else cyc
        dec = np.concatenate([cyc, ex], 1) if mode in ("future", "dec", "ch") else cyc
        self.m_enc = torch.tensor(enc, dtype=torch.float32, device=device)
        self.m_dec = torch.tensor(dec, dtype=torch.float32, device=device)

    def inverse(self, z):
        v = z * self.sd + self.mu
        if self.transform == "log1p":
            v = np.expm1(v)
        return np.clip(v, 0, None)


def origins(start, stop_incl, stride):
    o = list(range(start, stop_incl + 1, stride))
    if o and o[-1] != stop_incl:
        o.append(stop_incl)
    return np.array(o, dtype=np.int64)


def make_batch(prep, o, args, device, with_target=True):
    o = torch.as_tensor(o, device=device)
    enc_idx = o[:, None] + torch.arange(-args.seq_len, 0, device=device)
    dec_idx = o[:, None] + torch.arange(-args.label_len, args.pred_len, device=device)
    x_enc = prep.z[enc_idx].unsqueeze(-1)
    m_enc = prep.m_enc[enc_idx]
    m_dec = prep.m_dec[dec_idx]
    if not with_target:
        return x_enc, m_enc, m_dec, None
    tgt_idx = o[:, None] + torch.arange(0, args.pred_len, device=device)
    return x_enc, m_enc, m_dec, prep.z[tgt_idx].unsqueeze(-1)


## Metrics and baselines (original units)

In [ ]:
def metrics(y, p):
    y, p = np.asarray(y, float).ravel(), np.asarray(p, float).ravel()
    den = np.abs(y) + np.abs(p)
    sm = np.where(den > 0, 2 * np.abs(p - y) / np.where(den > 0, den, 1), 0.0)
    return {"MAE": float(np.mean(np.abs(y - p))),
            "RMSE": float(np.sqrt(np.mean((y - p) ** 2))),
            "sMAPE": float(100 * np.mean(sm))}


def baselines(y, val_o, pred_len, periods):
    tgt = np.stack([y[o:o + pred_len] for o in val_o])
    out = {"persistence": metrics(tgt, np.stack([np.full(pred_len, y[o - 1]) for o in val_o]))}
    for p in sorted(set(periods + [pred_len])):
        fc = np.stack([[y[o - p + (h % p)] for h in range(pred_len)] for o in val_o])
        out[f"seasonal_naive_{p}"] = metrics(tgt, fc)
    return out


## Training / evaluation

In [ ]:
def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def build_model(prep, args, device):
    n_cov = prep.n_cov if prep.mode == "ch" else 0
    m = Autoformer(args.seq_len, args.label_len, args.pred_len,
                   prep.m_enc.shape[1], prep.m_dec.shape[1],
                   d_model=args.d_model, n_heads=args.n_heads, e_layers=args.e_layers,
                   d_layers=args.d_layers, d_ff=args.d_ff, moving_avg=args.moving_avg,
                   factor=args.factor, dropout=args.dropout, n_cov=n_cov).to(device)
    m.revin = getattr(args, "revin", True)
    return m


@torch.no_grad()
def predict_scaled(model, prep, o, args, device, with_target=True):
    model.eval()
    preds, tgts = [], []
    for i in range(0, len(o), args.batch_size):
        x, me, md, t = make_batch(prep, o[i:i + args.batch_size], args, device, with_target)
        preds.append(model(x, me, md).squeeze(-1).cpu().numpy())
        if with_target:
            tgts.append(t.squeeze(-1).cpu().numpy())
    return np.concatenate(preds), (np.concatenate(tgts) if with_target else None)


def train(prep, args, seed, device, train_o, val_o, max_epochs, patience, tag=""):
    set_seed(seed)
    model = build_model(prep, args, device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=getattr(args, "wd", 0.0))
    use_swa = getattr(args, "swa", False)          # average weights over epochs (no extra params)
    swa = torch.optim.swa_utils.AveragedModel(model) if use_swa else None
    rng = np.random.default_rng(seed)
    best, best_state, best_ep, hist, ep_run = float("inf"), None, 0, [], 0
    for ep in range(1, max_epochs + 1):
        for g in opt.param_groups:
            g["lr"] = args.lr * 0.5 ** ((ep - 1) // args.lr_decay_every)
        model.train()
        t0, tot, nb = time.time(), 0.0, 0
        perm = rng.permutation(train_o)
        for i in range(0, len(perm), args.batch_size):
            x, me, md, t = make_batch(prep, perm[i:i + args.batch_size], args, device)
            loss = F.mse_loss(model(x, me, md), t)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot, nb = tot + loss.item(), nb + 1
        if use_swa:
            swa.update_parameters(model)
        ev = swa.module if use_swa else model
        ep_run = ep
        msg = f"{tag} ep {ep:2d} train_mse {tot / max(nb, 1):.4f} ({time.time() - t0:.0f}s)"
        if len(val_o):
            p, t = predict_scaled(ev, prep, val_o, args, device)
            vl = float(np.mean((p - t) ** 2))
            vr = metrics(prep.inverse(t), prep.inverse(p))["RMSE"]
            hist.append({"epoch": ep, "val_mse_scaled": vl, "val_rmse": vr})
            msg += f" val_mse {vl:.4f} val_RMSE {vr:.3f}"
            if vl < best:
                best, best_ep = vl, ep
                best_state = {k: v.detach().clone() for k, v in ev.state_dict().items()}
            elif patience > 0 and ep - best_ep >= patience:
                print(msg + "  -> early stop")
                break
        print(msg)
    final = swa.module if use_swa else model
    if patience > 0 and best_state is not None:
        final.load_state_dict(best_state)
    return final, ep_run, best_ep, hist


def setup(args):
    device = torch.device(args.device if args.device != "auto"
                          else ("cuda" if torch.cuda.is_available() else "cpu"))
    y, X, cols, tidx = load_raw(args)
    if args.periods == "auto":
        found = detect_periods(y[: len(y) - args.val_len])
        periods = [p for p, _ in found]
        print(f"[periods] auto-detected (period, prominence): {found}")
    elif args.periods == "none":
        periods = []
    else:
        periods = [int(p) for p in args.periods.split(",")]
    print(f"[setup] device={device}  n={len(y)}  covariates={cols}  periods={periods}")
    return device, y, X, cols, tidx, periods


## Commands

In [ ]:
def cmd_analyze(args):
    y, X, cols, tidx = load_raw(args)
    s = pd.Series(y)
    print("== target ==")
    print(s.describe(percentiles=[.01, .25, .5, .75, .99]).to_string())
    print(f"skew {s.skew():.2f}   zeros {(s == 0).mean() * 100:.2f}%")
    print("ACF:", {l: round(s.autocorr(l), 3) for l in [1, 2, 3, 6, 12, 24, 48, 72, 168, 336]})
    ls = np.log1p(s.clip(lower=0))
    print("ACF (log1p, after 337-step detrend):",
          {l: round((ls - ls.rolling(337, center=True, min_periods=1).mean()).autocorr(l), 3)
           for l in [12, 24, 48, 84, 168, 336]})
    print("periodogram peaks (period, prominence):", detect_periods(y, top=5))
    print("\n== covariates ==")
    n = len(y)
    dy = np.diff(np.log1p(np.clip(y, 0, None)))
    for j, c in enumerate(cols):
        col = X[:, j]
        kind = "binary" if set(np.unique(col)).issubset({0.0, 1.0}) else "continuous"
        r0 = np.corrcoef(col[:n], y)[0, 1]
        rlog = np.corrcoef(col[:n], np.log1p(np.clip(y, 0, None)))[0, 1]
        rd = np.corrcoef(np.diff(col[:n]), dy)[0, 1] if np.std(np.diff(col[:n])) > 0 else float("nan")
        print(f"{c:>14s} {kind:>10s} mean {col.mean():9.3f}  corr(y) {r0:+.3f}  "
              f"corr(log y) {rlog:+.3f}  corr(dx, dlog y) {rd:+.3f}")


def cmd_experiment(args):
    device, y, X, cols, tidx, periods = setup(args)
    modes = [m.strip() for m in args.exog_modes.split(",")]
    seeds = [int(s) for s in args.seeds.split(",")]
    os.makedirs(args.out_dir, exist_ok=True)
    rows, horizon = [], {}
    base_printed = False
    for mode in modes:
        prep = Prep(y, X, cols, tidx, periods, args, mode, device)
        train_o = origins(args.seq_len, prep.fit_end - args.pred_len, args.train_stride)
        val_o = origins(prep.fit_end, prep.n - args.pred_len, args.val_stride)
        if not base_printed:
            print(f"[split] train origins {len(train_o)} (targets < {prep.fit_end}), "
                  f"val origins {len(val_o)} (targets in [{prep.fit_end}, {prep.n}))")
            bl = baselines(y, val_o, args.pred_len, periods)
            for k, v in bl.items():
                rows.append({"model": k, "seed": -1, **v, "params": 0, "epochs_run": 0, "best_epoch": 0})
                print(f"[baseline] {k:22s} " + "  ".join(f"{m} {x:.3f}" for m, x in v.items()))
            base_printed = True
        for seed in seeds:
            t0 = time.time()
            model, ep_run, best_ep, hist = train(prep, args, seed, device, train_o, val_o,
                                                 args.epochs, args.patience, tag=f"[{mode}|s{seed}]")
            p, t = predict_scaled(model, prep, val_o, args, device)
            P, T = prep.inverse(p), prep.inverse(t)
            m = metrics(T, P)
            horizon.setdefault(mode, []).append(np.sqrt(np.mean((P - T) ** 2, axis=0)))
            rows.append({"model": f"autoformer_{mode}", "seed": seed, **m, "params": count_params(model),
                         "epochs_run": ep_run, "best_epoch": best_ep, "fit_sec": round(time.time() - t0, 1)})
            print(f"[{mode}|s{seed}] VAL " + "  ".join(f"{k} {v:.3f}" for k, v in m.items())
                  + f"  P={count_params(model)} best_ep={best_ep}")
            pd.DataFrame(rows).to_csv(os.path.join(args.out_dir, "experiment_runs.csv"), index=False)

    df = pd.DataFrame(rows)
    nn_df = df[df.seed >= 0]
    summ = nn_df.groupby("model").agg(
        RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
        sMAPE_mean=("sMAPE", "mean"), sMAPE_std=("sMAPE", "std"),
        params=("params", "first"), best_epoch_median=("best_epoch", "median"), runs=("seed", "count"))
    print("\n== validation summary (mean / std across seeds, original units) ==")
    print(summ.round(3).to_string())
    summ.to_csv(os.path.join(args.out_dir, "experiment_summary.csv"))

    # paired (same-seed) differences against the no-covariate model -> the ablation
    if "none" in modes:
        ref = nn_df[nn_df.model == "autoformer_none"].set_index("seed")
        print("\n== paired ablation vs. no covariates (negative = better) ==")
        for mode in modes:
            if mode == "none":
                continue
            cur = nn_df[nn_df.model == f"autoformer_{mode}"].set_index("seed")
            for met in ["RMSE", "MAE", "sMAPE"]:
                d = (cur[met] - ref[met]).dropna()
                print(f"  {mode:>7s} {met:5s}: mean diff {d.mean():+.3f}  std {d.std():.3f}  "
                      f"wins {(d < 0).sum()}/{len(d)}")

    pd.DataFrame({m: np.mean(v, 0) for m, v in horizon.items()}).rename_axis("h").to_csv(
        os.path.join(args.out_dir, "horizon_rmse.csv"))
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7, 3.5))
        for m, v in horizon.items():
            ax.plot(np.arange(1, args.pred_len + 1), np.mean(v, 0), label=f"Autoformer ({m})")
        ax.set_xlabel("forecast step h")
        ax.set_ylabel("validation RMSE")
        ax.legend()
        ax.grid(alpha=.3)
        fig.tight_layout()
        fig.savefig(os.path.join(args.out_dir, "horizon_rmse.pdf"))
        print(f"[saved] {args.out_dir}/horizon_rmse.pdf")
    except Exception as e:  # plotting is optional
        print(f"[plot skipped] {e}")
    print(f"[saved] {args.out_dir}/experiment_runs.csv, experiment_summary.csv, horizon_rmse.csv")


def cmd_predict(args):
    device, y, X, cols, tidx, periods = setup(args)
    seeds = [int(s) for s in args.seeds.split(",")]
    prep = Prep(y, X, cols, tidx, periods, args, args.exog_mode, device)
    last = prep.n - args.pred_len
    if args.refit:      # train on every available window; no validation left
        train_o, val_o = origins(args.seq_len, last, args.train_stride), np.array([], dtype=np.int64)
    else:
        train_o = origins(args.seq_len, prep.fit_end - args.pred_len, args.train_stride)
        val_o = origins(prep.fit_end, last, args.val_stride)
    preds, P_total, E_total = [], 0, 0
    for seed in seeds:
        model, ep_run, best_ep, _ = train(prep, args, seed, device, train_o, val_o,
                                          args.epochs, args.patience, tag=f"[final|s{seed}]")
        p, _ = predict_scaled(model, prep, np.array([prep.n]), args, device, with_target=False)
        preds.append(prep.inverse(p[0]))
        P_total += count_params(model)
        E_total += ep_run                    # every epoch actually trained counts
    fc = np.mean(preds, axis=0)
    assert fc.shape == (args.pred_len,) and np.isfinite(fc).all()

    os.makedirs(args.out_dir, exist_ok=True)
    idx = np.arange(prep.n + 1, prep.n + args.pred_len + 1)
    pd.DataFrame({"time_idx": idx, "value": fc}).to_csv(os.path.join(args.out_dir, "forecast.csv"), index=False)
    line = ",".join(f"{v:.4f}" for v in fc)
    with open(os.path.join(args.out_dir, "forecast_line.txt"), "w") as f:
        f.write(line + "\n")
    info = {"P_trainable_params": P_total, "E_training_epochs": E_total, "seeds": seeds,
            "first_time_idx": int(idx[0]), "last_time_idx": int(idx[-1]), "periods": periods,
            "config": {k: v for k, v in vars(args).items() if k != "func"}}
    with open(os.path.join(args.out_dir, "submission_info.json"), "w") as f:
        json.dump(info, f, indent=2)
    print(f"\nForecast for time_idx {idx[0]}..{idx[-1]} ({len(fc)} values) -> {args.out_dir}/forecast_line.txt")
    print(f"DECLARE  P = {P_total}   E = {E_total}")
    print("\nPaste into the leaderboard:\n" + line)


## Config (final config)

In [ ]:
from types import SimpleNamespace
args = SimpleNamespace(
    data_dir="data", train_file="student_train.csv", external_file="optional_external_data.csv",
    features="all", periods="auto", target_transform="none",
    seq_len=336, label_len=96, pred_len=168,
    d_model=16, n_heads=4, e_layers=1, d_layers=1, d_ff=32, moving_avg=25, factor=1.0, dropout=0.05,
    lr=1e-3, lr_decay_every=3, batch_size=64, epochs=10, patience=0,
    train_stride=4,          # 4 = ~4x faster on CPU
    val_len=168*12, val_stride=24, device="auto", out_dir="outputs",
    exog_modes="none,past,future", seeds="0,1,2",   # experiment
    exog_mode="future", refit=True,                # predict
)


## 1. Data analysis

In [ ]:
cmd_analyze(args)


== target ==
count    43656.000000
mean        98.173539
std         90.834754
min          0.000000
1%           6.000000
25%         30.000000
50%         73.000000
75%        136.000000
99%        418.000000
max        994.000000
skew 1.83   zeros 0.00%
ACF: {1: np.float64(0.966), 2: np.float64(0.92), 3: np.float64(0.874), 6: np.float64(0.749), 12: np.float64(0.57), 24: np.float64(0.402), 48: np.float64(0.158), 72: np.float64(0.077), 168: np.float64(0.028), 336: np.float64(0.048)}
ACF (log1p, after 337-step detrend): {12: np.float64(0.503), 24: np.float64(0.257), 48: np.float64(-0.04), 84: np.float64(-0.184), 168: np.float64(-0.056), 336: np.float64(0.059)}
periodogram peaks (period, prominence): [(24, 232.40049964054109), (12, 159.49901483288613), (6, 57.30021038487577), (5, 16.066138144429008), (386, 12.6861756296405)]

== covariates ==
     feature_A continuous mean     1.817  corr(y) +0.169  corr(log y) +0.308  corr(dx, dlog y) +0.183
     feature_B continuous mean    12.449  co

In [ ]:
import pandas as pd
SH = [("feature_G", 4), ("feature_H", 5), ("feature_I", 7), ("feature_J", 3)]
grid = {
    "cur":        {},
    "shift":      {"shifts": SH},
    "roll":       {"roll_bin": [6, 24]},
    "shift_roll": {"shifts": SH, "roll_bin": [6, 24]},
    "ch_both":    {"exog_modes": "ch", "shifts": SH, "roll_bin": [6, 24]},
}
res = []
for name, ov in grid.items():
    a = SimpleNamespace(**{**vars(args), "exog_modes": "future", "seeds": "0,1,2",
                           "patience": 0, "refit": False, "out_dir": f"grid2/{name}", **ov})
    print(f"\n######## {name} ########")
    cmd_experiment(a)
    s = pd.read_csv(f"grid2/{name}/experiment_summary.csv"); s["cfg"] = name; res.append(s)
pd.concat(res)[["cfg", "RMSE_mean", "RMSE_std", "MAE_mean", "params"]].sort_values("RMSE_mean")


######## cur ########
[periods] auto-detected (period, prominence): [(24, 168.30822306908408), (12, 138.29275001501827)]
[setup] device=cuda  n=43656  covariates=['feature_A', 'feature_B', 'feature_C', 'feature_D', 'feature_E', 'feature_F', 'feature_G', 'feature_H', 'feature_I', 'feature_J']  periods=[24, 12]
[split] train origins 10285 (targets < 41640), val origins 78 (targets in [41640, 43656))
[baseline] persistence            MAE 123.864  RMSE 166.385  sMAPE 102.242
[baseline] seasonal_naive_12      MAE 118.728  RMSE 160.139  sMAPE 105.686
[baseline] seasonal_naive_24      MAE 118.940  RMSE 158.639  sMAPE 106.207
[baseline] seasonal_naive_168     MAE 126.068  RMSE 165.867  sMAPE 109.336
[future|s0] ep  1 train_mse 0.8354 (3s) val_mse 1.0352 val_RMSE 89.284
[future|s0] ep  2 train_mse 0.6856 (3s) val_mse 0.9858 val_RMSE 88.092
[future|s0] ep  3 train_mse 0.6358 (4s) val_mse 0.9733 val_RMSE 86.674
[future|s0] ep  4 train_mse 0.6033 (3s) val_mse 0.9546 val_RMSE 85.572
[future|s0] ep

,cfg,RMSE_mean,RMSE_std,MAE_mean,params
0,ch_both,81.586828,0.662504,57.361306,7777
0,shift,82.736876,2.065684,56.578243,6113
0,shift_roll,84.204441,3.107271,59.312915,6369
0,roll,85.900595,1.422761,60.616935,6241
0,cur,85.957108,1.077653,59.225720,5985


## External-data ablation: none / past / future (report Table 2)
Final configuration, 3 seeds, 10 epochs, no early stopping

In [ ]:
cmd_experiment(args)


[periods] auto-detected (period, prominence): [(24, 168.30822306908408), (12, 138.29275001501827)]
[setup] device=cuda  n=43656  covariates=['feature_A', 'feature_B', 'feature_C', 'feature_D', 'feature_E', 'feature_F', 'feature_G', 'feature_H', 'feature_I', 'feature_J']  periods=[24, 12]
[split] train origins 10285 (targets < 41640), val origins 78 (targets in [41640, 43656))
[baseline] persistence            MAE 123.864  RMSE 166.385  sMAPE 102.242
[baseline] seasonal_naive_12      MAE 118.728  RMSE 160.139  sMAPE 105.686
[baseline] seasonal_naive_24      MAE 118.940  RMSE 158.639  sMAPE 106.207
[baseline] seasonal_naive_168     MAE 126.068  RMSE 165.867  sMAPE 109.336
[none|s0] ep  1 train_mse 1.0853 (3s) val_mse 1.8289 val_RMSE 121.229
[none|s0] ep  2 train_mse 1.0687 (4s) val_mse 1.8281 val_RMSE 121.202
[none|s0] ep  3 train_mse 1.0637 (3s) val_mse 1.8255 val_RMSE 121.116
[none|s0] ep  4 train_mse 1.0607 (3s) val_mse 1.8240 val_RMSE 121.066
[none|s0] ep  5 train_mse 1.0576 (3s) val

## Submission 1 — final, best leaderboard score (RMSE 92.30)
Final config, seed 0, refit on the full history for 10 epochs. Declared **P = 5985, E = 10**.

In [ ]:
final = SimpleNamespace(**{**vars(args), "seeds": "0"})
cmd_predict(final)

[periods] auto-detected (period, prominence): [(24, 168.30822306908408), (12, 138.29275001501827)]
[setup] device=cuda  n=43656  covariates=['feature_A', 'feature_B', 'feature_C', 'feature_D', 'feature_E', 'feature_F', 'feature_G', 'feature_H', 'feature_I', 'feature_J']  periods=[24, 12]
[final|s0] ep  1 train_mse 0.8394 (4s)
[final|s0] ep  2 train_mse 0.6861 (3s)
[final|s0] ep  3 train_mse 0.6380 (3s)
[final|s0] ep  4 train_mse 0.6097 (4s)
[final|s0] ep  5 train_mse 0.6010 (3s)
[final|s0] ep  6 train_mse 0.5926 (3s)
[final|s0] ep  7 train_mse 0.5816 (3s)
[final|s0] ep  8 train_mse 0.5776 (4s)
[final|s0] ep  9 train_mse 0.5767 (3s)
[final|s0] ep 10 train_mse 0.5720 (3s)

Forecast for time_idx 43657..43824 (168 values) -> outputs/forecast_line.txt
DECLARE  P = 5985   E = 10

Paste into the leaderboard:
49.8637,48.9552,40.4063,44.9609,34.2495,35.8450,35.3122,32.7831,33.1688,28.0312,24.7805,23.7343,24.8445,26.9133,31.0516,31.4184,37.9922,46.6704,63.7211,79.0437,89.2101,97.6609,94.7911,93.

## Ensembles (report 2.8)
### Five-seed ensemble of the final config (validation)

In [ ]:
def ens_val(a, seeds):
    device, y, X, cols, tidx, periods = setup(a)
    prep = Prep(y, X, cols, tidx, periods, a, a.exog_mode, device)
    tr = origins(a.seq_len, prep.fit_end - a.pred_len, a.train_stride)
    va = origins(prep.fit_end, prep.n - a.pred_len, a.val_stride)
    preds = []
    for s in seeds:
        m, _, _, _ = train(prep, a, s, device, tr, va, a.epochs, 0, tag=f"[ens|s{s}]")
        p, t = predict_scaled(m, prep, va, a, device)
        preds.append(prep.inverse(p))
        r = metrics(prep.inverse(t), np.mean(preds, 0))
        print(f">>> {len(preds)} seed(s) averaged: RMSE {r['RMSE']:.2f}  MAE {r['MAE']:.2f}")

ens_val(SimpleNamespace(**{**vars(args), "refit": False}), [0, 1, 2, 3, 4])

[periods] auto-detected (period, prominence): [(24, 168.30822306908408), (12, 138.29275001501827)]
[setup] device=cuda  n=43656  covariates=['feature_A', 'feature_B', 'feature_C', 'feature_D', 'feature_E', 'feature_F', 'feature_G', 'feature_H', 'feature_I', 'feature_J']  periods=[24, 12]
[ens|s0] ep  1 train_mse 0.8351 (6s) val_mse 1.0375 val_RMSE 89.332
[ens|s0] ep  2 train_mse 0.6811 (3s) val_mse 1.0031 val_RMSE 88.765
[ens|s0] ep  3 train_mse 0.6280 (3s) val_mse 0.9671 val_RMSE 86.731
[ens|s0] ep  4 train_mse 0.5986 (4s) val_mse 0.9145 val_RMSE 83.698
[ens|s0] ep  5 train_mse 0.5858 (3s) val_mse 0.8986 val_RMSE 83.772
[ens|s0] ep  6 train_mse 0.5818 (3s) val_mse 0.9069 val_RMSE 83.777
[ens|s0] ep  7 train_mse 0.5709 (3s) val_mse 0.8683 val_RMSE 80.859
[ens|s0] ep  8 train_mse 0.5691 (3s) val_mse 0.8686 val_RMSE 81.831
[ens|s0] ep  9 train_mse 0.5660 (3s) val_mse 0.8598 val_RMSE 80.727
[ens|s0] ep 10 train_mse 0.5649 (3s) val_mse 0.8628 val_RMSE 80.786
>>> 1 seed(s) averaged: RMSE 80

### Five-seed forecast (produced, not submitted)

In [ ]:
final = SimpleNamespace(**{**vars(args), "seeds": "0,1,2,3,4"})
cmd_predict(final)

[periods] auto-detected (period, prominence): [(24, 168.30822306908408), (12, 138.29275001501827)]
[setup] device=cuda  n=43656  covariates=['feature_A', 'feature_B', 'feature_C', 'feature_D', 'feature_E', 'feature_F', 'feature_G', 'feature_H', 'feature_I', 'feature_J']  periods=[24, 12]
[final|s0] ep  1 train_mse 0.8399 (4s)
[final|s0] ep  2 train_mse 0.6856 (3s)
[final|s0] ep  3 train_mse 0.6317 (4s)
[final|s0] ep  4 train_mse 0.6054 (3s)
[final|s0] ep  5 train_mse 0.5943 (3s)
[final|s0] ep  6 train_mse 0.5850 (3s)
[final|s0] ep  7 train_mse 0.5754 (3s)
[final|s0] ep  8 train_mse 0.5712 (3s)
[final|s0] ep  9 train_mse 0.5697 (3s)
[final|s0] ep 10 train_mse 0.5650 (4s)
[final|s1] ep  1 train_mse 0.8647 (3s)
[final|s1] ep  2 train_mse 0.6846 (3s)
[final|s1] ep  3 train_mse 0.6425 (3s)
[final|s1] ep  4 train_mse 0.6106 (4s)
[final|s1] ep  5 train_mse 0.5972 (3s)
[final|s1] ep  6 train_mse 0.5854 (3s)
[final|s1] ep  7 train_mse 0.5762 (3s)
[final|s1] ep  8 train_mse 0.5724 (4s)
[final|s1

### Mixed ensemble: 5 `future` + 5 `ch` models (validation), then submission 2
Submission 2: test RMSE 93.03, P = 63050, E = 100.

In [ ]:
def ens_val_mix(runs):
    preds = []
    for ov, s in runs:
        a = SimpleNamespace(**{**vars(args), "refit": False, **ov})
        device, y, X, cols, tidx, periods = setup(a)
        prep = Prep(y, X, cols, tidx, periods, a, a.exog_mode, device)
        tr = origins(a.seq_len, prep.fit_end - a.pred_len, a.train_stride)
        va = origins(prep.fit_end, prep.n - a.pred_len, a.val_stride)
        m, _, _, _ = train(prep, a, s, device, tr, va, a.epochs, 0, tag=f"[{a.exog_mode}|s{s}]")
        p, t = predict_scaled(m, prep, va, a, device)
        preds.append(prep.inverse(p))
        single = metrics(prep.inverse(t), preds[-1])["RMSE"]
        r = metrics(prep.inverse(t), np.mean(preds, 0))
        print(f">>> model {len(preds)} ({a.exog_mode}, seed {s}) alone {single:.2f} | average of {len(preds)}: RMSE {r['RMSE']:.2f}")


def predict_mix(runs):
    fcs, P, E = [], 0, 0
    for ov, s in runs:
        a = SimpleNamespace(**{**vars(args), "refit": True, **ov})
        device, y, X, cols, tidx, periods = setup(a)
        prep = Prep(y, X, cols, tidx, periods, a, a.exog_mode, device)
        tr = origins(a.seq_len, prep.n - a.pred_len, a.train_stride)
        m, ep, _, _ = train(prep, a, s, device, tr, np.array([], dtype=np.int64), a.epochs, 0,
                            tag=f"[final {a.exog_mode}|s{s}]")
        p, _ = predict_scaled(m, prep, np.array([prep.n]), a, device, with_target=False)
        fcs.append(prep.inverse(p[0]))
        P += count_params(m)
        E += ep
    fc = np.mean(fcs, 0)
    assert fc.shape == (168,) and np.isfinite(fc).all()
    line = ",".join(f"{v:.4f}" for v in fc)
    os.makedirs(args.out_dir, exist_ok=True)
    with open(os.path.join(args.out_dir, "forecast_line_mix.txt"), "w") as f:
        f.write(line + "\n")
    print(f"\nDECLARE  P = {P}   E = {E}\n\nPaste into the leaderboard:\n{line}")


runs = [({"exog_mode": "future"}, s) for s in range(5)] + [({"exog_mode": "ch"}, s) for s in range(5)]

In [ ]:
ens_val_mix(runs)

[periods] auto-detected (period, prominence): [(24, 168.30822306908408), (12, 138.29275001501827)]
[setup] device=cuda  n=43656  covariates=['feature_A', 'feature_B', 'feature_C', 'feature_D', 'feature_E', 'feature_F', 'feature_G', 'feature_H', 'feature_I', 'feature_J']  periods=[24, 12]
[future|s0] ep  1 train_mse 0.8351 (5s) val_mse 1.0370 val_RMSE 89.310
[future|s0] ep  2 train_mse 0.6834 (4s) val_mse 0.9718 val_RMSE 86.621
[future|s0] ep  3 train_mse 0.6297 (4s) val_mse 1.0095 val_RMSE 88.161
[future|s0] ep  4 train_mse 0.6009 (4s) val_mse 0.9806 val_RMSE 86.430
[future|s0] ep  5 train_mse 0.5847 (3s) val_mse 0.9516 val_RMSE 85.805
[future|s0] ep  6 train_mse 0.5830 (4s) val_mse 0.9492 val_RMSE 85.275
[future|s0] ep  7 train_mse 0.5734 (3s) val_mse 0.9346 val_RMSE 83.915
[future|s0] ep  8 train_mse 0.5720 (3s) val_mse 0.9303 val_RMSE 84.688
[future|s0] ep  9 train_mse 0.5675 (4s) val_mse 0.9229 val_RMSE 83.627
[future|s0] ep 10 train_mse 0.5649 (3s) val_mse 0.9197 val_RMSE 83.867
>

In [ ]:
predict_mix(runs)

[periods] auto-detected (period, prominence): [(24, 168.30822306908408), (12, 138.29275001501827)]
[setup] device=cuda  n=43656  covariates=['feature_A', 'feature_B', 'feature_C', 'feature_D', 'feature_E', 'feature_F', 'feature_G', 'feature_H', 'feature_I', 'feature_J']  periods=[24, 12]
[final future|s0] ep  1 train_mse 0.8394 (6s)
[final future|s0] ep  2 train_mse 0.6891 (4s)
[final future|s0] ep  3 train_mse 0.6397 (3s)
[final future|s0] ep  4 train_mse 0.6115 (4s)
[final future|s0] ep  5 train_mse 0.6027 (3s)
[final future|s0] ep  6 train_mse 0.5933 (3s)
[final future|s0] ep  7 train_mse 0.5841 (4s)
[final future|s0] ep  8 train_mse 0.5817 (3s)
[final future|s0] ep  9 train_mse 0.5798 (3s)
[final future|s0] ep 10 train_mse 0.5761 (3s)
[periods] auto-detected (period, prominence): [(24, 168.30822306908408), (12, 138.29275001501827)]
[setup] device=cuda  n=43656  covariates=['feature_A', 'feature_B', 'feature_C', 'feature_D', 'feature_E', 'feature_F', 'feature_G', 'feature_H', 'featu

###Lag-aware features (report Table 6)
Shifted indicator copies (`shift`), share of recent steps in each indicator state (`roll`), and input-channel routing (`ch`).

In [ ]:
def lag_report(a, max_lag=72):
    """Correlation between target at t and feature at t-k (k>0: feature LEADS target)."""
    y, X, cols, _ = load_raw(a)
    n = len(y)
    ly = np.log1p(np.clip(y, 0, None))
    dly = np.diff(ly)
    rows = []
    for j, c in enumerate(cols):
        x = X[:n, j]
        dx = np.diff(x)
        best_lvl, best_chg = (0, 0.0), (0, 0.0)
        for k in range(-max_lag, max_lag + 1):
            if k >= 0:
                a1, b1 = ly[k:], x[:n - k]
                a2, b2 = dly[k:], dx[:len(dx) - k]
            else:
                a1, b1 = ly[:n + k], x[-k:]
                a2, b2 = dly[:len(dly) + k], dx[-k:]
            r1 = np.corrcoef(a1, b1)[0, 1] if b1.std() > 0 else 0
            r2 = np.corrcoef(a2, b2)[0, 1] if b2.std() > 0 else 0
            if abs(r1) > abs(best_lvl[1]): best_lvl = (k, r1)
            if abs(r2) > abs(best_chg[1]): best_chg = (k, r2)
        r0 = np.corrcoef(ly, x)[0, 1] if x.std() > 0 else 0
        rows.append({"feature": c, "corr_lag0": round(r0, 3),
                     "best_lag_level": best_lvl[0], "corr_level": round(best_lvl[1], 3),
                     "best_lag_change": best_chg[0], "corr_change": round(best_chg[1], 3)})
    df = pd.DataFrame(rows)
    print("lag k > 0: feature at t-k relates to target at t (feature LEADS)\n"
          "lag k < 0: feature at t+|k| relates to target at t (feature LAGS)")
    print(df.to_string(index=False))
    return df

lag_report(args)

lag k > 0: feature at t-k relates to target at t (feature LEADS)
lag k < 0: feature at t+|k| relates to target at t (feature LAGS)
  feature  corr_lag0  best_lag_level  corr_level  best_lag_change  corr_change
feature_A      0.308               0       0.308                0        0.183
feature_B      0.019             -14       0.111                2       -0.097
feature_C     -0.159              -6      -0.172                6       -0.078
feature_D     -0.348               0      -0.348                0       -0.042
feature_E      0.032             -31       0.041               10        0.017
feature_F     -0.049              18      -0.079                1       -0.036
feature_G     -0.064               4      -0.072              -66       -0.016
feature_H     -0.346               5      -0.433                1       -0.050
feature_I      0.226               7       0.316                7        0.024
feature_J      0.182               3       0.204                2        0.020


,feature,corr_lag0,best_lag_level,corr_level,best_lag_change,corr_change
0,feature_A,0.308,0,0.308,0,0.183
1,feature_B,0.019,-14,0.111,2,-0.097
2,feature_C,-0.159,-6,-0.172,6,-0.078
3,feature_D,-0.348,0,-0.348,0,-0.042
4,feature_E,0.032,-31,0.041,10,0.017
5,feature_F,-0.049,18,-0.079,1,-0.036
6,feature_G,-0.064,4,-0.072,-66,-0.016
7,feature_H,-0.346,5,-0.433,1,-0.050
8,feature_I,0.226,7,0.316,7,0.024
9,feature_J,0.182,3,0.204,2,0.020


## Lag-feature ensemble (validation), then submission 3
`ch` + shifts + persistence shares, 5 seeds. Submission 3: test RMSE 94.22, P = 38885, E = 50.

In [ ]:
SH = [("feature_G", 4), ("feature_H", 5), ("feature_I", 7), ("feature_J", 3)]
CHB = {"exog_mode": "ch", "shifts": SH, "roll_bin": [6, 24]}
runs = [(CHB, s) for s in range(5)]

ens_val_mix(runs)      # about 3 min: validation number for the report
predict_mix(runs)      # about 3 min: the forecast to submit

[periods] auto-detected (period, prominence): [(24, 168.30822306908408), (12, 138.29275001501827)]
[setup] device=cuda  n=43656  covariates=['feature_A', 'feature_B', 'feature_C', 'feature_D', 'feature_E', 'feature_F', 'feature_G', 'feature_H', 'feature_I', 'feature_J']  periods=[24, 12]
[ch|s0] ep  1 train_mse 0.7717 (5s) val_mse 0.9823 val_RMSE 85.770
[ch|s0] ep  2 train_mse 0.6221 (4s) val_mse 0.9132 val_RMSE 83.451
[ch|s0] ep  3 train_mse 0.5786 (4s) val_mse 0.8725 val_RMSE 81.086
[ch|s0] ep  4 train_mse 0.5541 (3s) val_mse 0.9063 val_RMSE 82.710
[ch|s0] ep  5 train_mse 0.5428 (3s) val_mse 0.9482 val_RMSE 85.275
[ch|s0] ep  6 train_mse 0.5373 (4s) val_mse 0.8986 val_RMSE 82.554
[ch|s0] ep  7 train_mse 0.5281 (3s) val_mse 0.9065 val_RMSE 81.719
[ch|s0] ep  8 train_mse 0.5250 (3s) val_mse 0.9077 val_RMSE 82.702
[ch|s0] ep  9 train_mse 0.5219 (4s) val_mse 0.8816 val_RMSE 81.249
[ch|s0] ep 10 train_mse 0.5189 (4s) val_mse 0.8960 val_RMSE 81.039
>>> model 1 (ch, seed 0) alone 81.04 | av